In [1]:
import numpy as np
import pandas as pd

from mesh.load_meshes import load_meshes
from models import Nodes, Elements, Edges

nodes, elements = load_meshes()

### Physical parameters

In [2]:
EO = 1  # Plane wave amplitude
EPS0: float = 8.85e-12  # Diel. permittivity of vacuum
EPS1: float = 1.0  # Permittivity of air
MU0: float = 4 * np.pi * 1e-7  # Mag. permeability of vacuum
MU1: float = 1.0  # Permeability of the air
SIG1 = 0.0  # Conductivity of the air (S/m)
V = 1 / np.sqrt(EPS0 * MU0 * EPS1 * MU1)  # wave speed
F_HZ = 4e9  # Plane wave frequency
OMEGA = 2 * np.pi * F_HZ  # Angular frequency
LAMBDA = V / F_HZ  # Wavelength
K0: float = 2 * np.pi / LAMBDA  # Free space wave number
KPW = np.array([0, K0])  # Direction of the plane wave

### Compute Boundary Elements 

In [3]:
def find_boundary_edges(elements: Elements) -> Edges:
    """Edges that belong to exactly one triangle: the outer border of the mesh."""
    tri = elements.node_idxs  # (T, 3)

    # Triangle [a, b, c] has edges (a,b), (b,c), (c,a), in that direction.
    edges = np.concatenate([tri[:, [0, 1]], tri[:, [1, 2]], tri[:, [2, 0]]])  # (3T, 2)

    # Two neighbours store their shared edge as (a,b) and (b,a).
    # Sorting each row makes both read (min, max), so they compare equal.
    sorted_edges = np.sort(edges, axis=1)

    # Count each distinct edge. `inverse` maps every row of `edges`
    # to its distinct edge, so counts[inverse] is a per-row count.
    # Interior edges appear twice, border edges once.
    _, inverse, counts = np.unique(
        sorted_edges, axis=0, return_inverse=True, return_counts=True
    )
    on_border = counts[inverse] == 1

    # Mask the UNSORTED edges, so the direction from the triangles survives.
    return Edges(node_idxs=edges[on_border])


boundary: Edges = find_boundary_edges(elements)  # air box + PEC slot edges

# Remove the air-box edges: their midpoint lies on one of the box's sides.
coords = nodes.xy[boundary.node_idxs]  # (M, 2 ends, 2 coords)
mid = coords.mean(axis=1)  # (M, 2)
box_min = nodes.xy.min(axis=0)  # (2,) = (x_min, y_min); taken from the mesh,
box_max = nodes.xy.max(axis=0)  # (2,) = (x_max, y_max);
on_box = (np.isclose(mid, box_min) | np.isclose(mid, box_max)).any(axis=1)
boundary: Edges = boundary[~on_box]  # PEC slot edges only, clockwise per slot